<a href="https://colab.research.google.com/github/vmware/versatile-data-kit/blob/main/examples/incremental-ingest-from-db-example-notebook/incremental-ingest-example-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Incremental Ingestion using Job Properties

This notebook provides a guide on how to perform incremental ingestion from a local SQLite database using [Versatile Data Kit (VDK)](https://github.com/vmware/versatile-data-kit). We will use job properties to track state between job runs and only ingest new records.

<a name="prerequisites"></a>
## 1. Prerequisites

### 1.1 Good to Know Before You Start

This tutorial can be easily understood if you are familiar with:

- **Python and SQL**: Basic commands and queries
- **Tools**: Comfort with Jupyter Notebook
- **VDK Concepts**: Understanding of [Data Jobs](https://github.com/vmware/versatile-data-kit/wiki/dictionary#data-job) and [Job Properties](https://github.com/vmware/versatile-data-kit/wiki/dictionary#job-properties)

### 1.2 Useful notebook shortcuts

- Click the **Play icon** in the left gutter of the cell;
- Type **Cmd/Ctrl+Enter** to run the cell in place;
- Type **Shift+Enter** to run the cell and move focus to the next cell (adding one if none exists); or
- Type **Alt+Enter** to run the cell and insert a new code cell immediately below it.

There are additional options for running some or all cells in the **Runtime** menu on top.

### 1.3 Install Versatile Data Kit and required plugins

In [ ]:
!pip install vdk-ipython vdk-sqlite vdk-ingest-http --quiet

The relevant Data Job code is in the upcoming cells.

Alternatively, you can see the implementation of the data job [here](https://github.com/vmware/versatile-data-kit/tree/main/examples/incremental-ingest-from-db-example).

## 2. Understanding Incremental Ingestion

Incremental ingestion is a data loading pattern where only new or changed records are ingested from a source to a target, rather than loading the entire dataset each time. This approach is useful when:

- The source dataset is large and loading it completely every time is inefficient
- You want to minimize resource usage and processing time
- You need to maintain a history of changes

In VDK, we can achieve this using **Job Properties** - a key-value store that persists between job runs.

## 3. Setup

### 3.1 Create the Source Database

First, we'll create a SQLite database with sample data that simulates a real-world scenario where new records are added over time.

In [ ]:
import sqlite3
import os

# Create the database directory
os.makedirs('data', exist_ok=True)

# Create the source database with initial data
source_db_path = 'data/source_example.db'
conn = sqlite3.connect(source_db_path)
cursor = conn.cursor()

# Create a table with sample data
cursor.execute('''
    CREATE TABLE IF NOT EXISTS increm_ingest (
        id INTEGER PRIMARY KEY,
        descr TEXT,
        reported_date TEXT
    )
''')

# Insert initial sample data
initial_data = [
    (1, 'record one', '2021-10-01'),
    (2, 'second record', '2021-10-02'),
    (3, 'this is record 3', '2021-10-03')
]

cursor.executemany('INSERT OR REPLACE INTO increm_ingest VALUES (?, ?, ?)', initial_data)
conn.commit()
conn.close()

print(f"Source database created at: {source_db_path}")
print("Initial data inserted successfully!")

Let's verify the initial data:

In [ ]:
# Verify initial data
conn = sqlite3.connect(source_db_path)
cursor = conn.cursor()
cursor.execute('SELECT * FROM increm_ingest ORDER BY reported_date')
rows = cursor.fetchall()
print("Initial data in source database:")
print("-" * 50)
for row in rows:
    print(f"ID: {row[0]}, Description: {row[1]}, Date: {row[2]}")
conn.close()

## 4. Configuration

Configure the environment variables needed for VDK to connect to the SQLite database:

In [ ]:
%env VDK_SQLITE_FILE=data/source_example.db
%env VDK_INGEST_TARGET_DEFAULT=data/target_example.db
%env VDK_DB_DEFAULT_TYPE=SQLITE

## 5. Load VDK Extension

In [ ]:
%reload_ext vdk.plugin.ipython

And load the VDK Job Control object:

In [ ]:
%reload_VDK

## 6. First Run - Initial Ingestion

Now let's implement the incremental ingestion logic. In the first run, since no previous state exists, all records will be ingested.

### Step 1: Create the target table and perform initial ingestion

In [ ]:
%%vdksql
DROP TABLE IF EXISTS incremental_ingest_from_db_example

In [ ]:
%%vdksql
CREATE TABLE incremental_ingest_from_db_example (
    id INTEGER,
    descr TEXT,
    reported_date TEXT
)

### Step 2: Implement the incremental ingestion logic

In [ ]:
# Get job_input object
job_input = vdk.get_job_input()

# Get last_date property:
# - If this is the first job run, initialize last_date to 1900-01-01 to fetch all rows
# - If the data job was run previously, use the property value stored from the previous run
last_date = job_input.get_property("last_date", "1900-01-01")

print(f"Using last_date: {last_date}")

# Select records from the source table that are newer than last_date
data = job_input.execute_query(
    f"""
    SELECT * FROM increm_ingest
    WHERE reported_date > '{last_date}'
    ORDER BY reported_date
    """
)

# Fetch table info containing the column names
table_info = job_input.execute_query("PRAGMA table_info(increm_ingest)")
column_names = [column[1] for column in table_info]

print(f"\nFound {len(data)} new records to ingest")
print(f"Column names: {column_names}")

In [ ]:
# If any data is returned, send the fetched records for ingestion
if len(data) > 0:
    job_input.send_tabular_data_for_ingestion(
        data,
        column_names=column_names,
        destination_table="incremental_ingest_from_db_example",
    )
    
    # Get the latest date from the ingested data
    latest_date = max(row[2] for row in data)
    
    # Update the last_date property
    job_input.set_all_properties({"last_date": latest_date})
    
    print(f"Successfully ingested {len(data)} rows.")
    print(f"Updated last_date property to: {latest_date}")
else:
    print("No new records to ingest.")

### Step 3: Verify the results of initial ingestion

In [ ]:
%%vdksql
SELECT * FROM incremental_ingest_from_db_example ORDER BY reported_date

## 7. Simulate New Data Being Added

Now let's simulate new records being added to the source database, as would happen in a real-world scenario:

In [ ]:
# Add new records to the source database
conn = sqlite3.connect(source_db_path)
cursor = conn.cursor()

new_data = [
    (4, "that's a new record!", '2021-11-01'),
    (5, 'and another new record..', '2021-11-02')
]

cursor.executemany('INSERT INTO increm_ingest VALUES (?, ?, ?)', new_data)
conn.commit()

# Show all data in source now
cursor.execute('SELECT * FROM increm_ingest ORDER BY reported_date')
rows = cursor.fetchall()
print("Updated source database (now has 5 records):")
print("-" * 50)
for row in rows:
    print(f"ID: {row[0]}, Description: {row[1]}, Date: {row[2]}")
conn.close()

## 8. Second Run - Incremental Ingestion

Now let's reload VDK and run the ingestion again. This time, it should only ingest the new records:

In [ ]:
%reload_VDK

In [ ]:
# Get job_input object again
job_input = vdk.get_job_input()

# Get the last_date from the previous run
last_date = job_input.get_property("last_date", "1900-01-01")

print(f"Retrieved last_date from previous run: {last_date}")

# Select only records newer than last_date
data = job_input.execute_query(
    f"""
    SELECT * FROM increm_ingest
    WHERE reported_date > '{last_date}'
    ORDER BY reported_date
    """
)

print(f"\nFound {len(data)} new records to ingest")

if len(data) > 0:
    for row in data:
        print(f"  - ID: {row[0]}, Date: {row[2]}")

In [ ]:
# Ingest the new records
if len(data) > 0:
    job_input.send_tabular_data_for_ingestion(
        data,
        column_names=column_names,
        destination_table="incremental_ingest_from_db_example",
    )
    
    # Update the last_date property
    latest_date = max(row[2] for row in data)
    job_input.set_all_properties({"last_date": latest_date})
    
    print(f"Successfully ingested {len(data)} new rows.")
    print(f"Updated last_date property to: {latest_date}")
else:
    print("No new records to ingest.")

## 9. Verify Final Results

Let's check the target table to see all the records, including the newly added ones:

In [ ]:
%%vdksql
SELECT * FROM incremental_ingest_from_db_example ORDER BY reported_date

## 10. Summary

In this notebook, we demonstrated:

1. **Creating a source database** with sample data
2. **Setting up VDK** with the necessary configuration
3. **Implementing incremental ingestion** using job properties to track the last processed date
4. **Simulating new data** being added to the source
5. **Running the ingestion again** to only process new records

### Key Concepts:

- **Job Properties**: VDK's `get_property()` and `set_all_properties()` methods allow you to store and retrieve state between job runs
- **Incremental Ingestion**: By tracking a timestamp or watermark, we can efficiently process only new or changed records
- **Tabular Data Ingestion**: The `send_tabular_data_for_ingestion()` method makes it easy to ingest query results into a target table

### Next Steps:

- Explore other [VDK examples](https://github.com/vmware/versatile-data-kit/wiki/Examples)
- Learn about [VDK Job Properties](https://github.com/vmware/versatile-data-kit/wiki/Job-Properties) for more advanced use cases
- Try creating your own incremental ingestion job with real data sources

## Additional Resources

- [VDK Wiki](https://github.com/vmware/versatile-data-kit/wiki)
- [Getting Started Guide](https://github.com/vmware/versatile-data-kit/wiki/Getting-Started)
- [Data Job Dictionary](https://github.com/vmware/versatile-data-kit/wiki/dictionary)
- [Incremental Ingestion Example (File-based)](https://github.com/vmware/versatile-data-kit/tree/main/examples/incremental-ingest-from-db-example)